# LLM Evaluation Methods
*Programmatic, human, and model-graded evaluation methods, a worked Recall@k example for a retriever, the five types of human evaluation, an LLM-as-judge grading example with Mean Absolute Error, and reference-based vs. reference-free evals*

## LLM eval methods: who actually judges the output

An **LLM eval method** is the mechanism used to decide whether a given output is good or not — the actual procedure that takes an output and produces a judgment about it. An eval pipeline determines *what* is being checked and *how success is defined*; the eval method is about *who or what carries out that check*.

There are three eval methods:

- **Programmatic (or deterministic).** A script or program determines the judgment — no human or model judgment involved.
- **Human.** A person reviews the output and judges it directly.
- **Model-graded (or LLM-as-judge).** Another LLM reviews the output and produces the judgment.

Any eval pipeline, regardless of what it's evaluating, is carried out by exactly one of these three.

```mermaid
flowchart TD
    A["LLM eval methods"] --> B["Programmatic /<br/>Deterministic"]
    A --> C["Human"]
    A --> D["Model-graded /<br/>LLM-as-judge"]
```

## A programmatic example: evaluating a retriever with Recall@k

Consider a RAG chatbot built on top of a knowledge base, meant to answer user questions without needing a person to reply manually. Evaluating it properly means building an eval pipeline for each component — starting here with the retriever.

**Task and target.** The task is checking whether the retriever works correctly; the target is that single component, not the full pipeline.

**Success criterion.** The retriever's job is to pull the most relevant documents from a vector database for a given query, so the natural metric is **Recall@k**: out of all the correct, relevant documents that exist for a query, how many did the system retrieve within its top *k* results?

$$\text{Recall@k} = \frac{|\{\text{relevant documents}\} \cap \{\text{top-}k\text{ retrieved documents}\}|}{|\{\text{relevant documents}\}|}$$

In words: the number of correct documents that show up in the top *k* retrieved results, divided by the total number of correct documents that actually exist for that query.

**Worked example.** Suppose the query is "What are the prerequisites for the ML course, and how long is it?" and, based on the underlying documents, the correct answer is known to live across exactly two documents — document 1001 and document 1003. So:

$$\{\text{relevant documents}\} = \{1001,\ 1003\}, \quad |\{\text{relevant documents}\}| = 2$$

The retriever is asked for its top 5 results (`k = 5`) and comes back with documents 1001, 1002, 1004, 1005, and 1006:

$$\{\text{top-}5\text{ retrieved documents}\} = \{1001,\ 1002,\ 1004,\ 1005,\ 1006\}$$

The overlap between the two sets is just document 1001, so:

$$\text{Recall@5} = \frac{|\{1001,\ 1003\} \cap \{1001,\ 1002,\ 1004,\ 1005,\ 1006\}|}{|\{1001,\ 1003\}|} = \frac{|\{1001\}|}{2} = \frac{1}{2} = 50\%$$

A perfect retriever would score 100%; the score can never exceed 100% or fall below 0%.

## Building the golden dataset for retrieval

Measuring recall at scale requires a dataset of questions paired with the documents that actually contain their answers. A practical way to build this: sample 50 to 100 realistic questions users might plausibly ask, covering a mix of easy, difficult, and edge-case queries — then have a human expert go through each question, search the underlying document set, and record exactly which document IDs contain the correct answer.

This is the golden dataset for the retriever evaluation specifically — not raw conversation logs, but a curated set of (question, correct document IDs) pairs, built once and reused every time the retriever is evaluated going forward.

## Running the evaluation programmatically

With the dataset in place, each question is sent through the retriever alone — not the full RAG pipeline, just the retriever — and the top-*k* documents it returns are recorded. For every question, recall is calculated by comparing the retrieved document IDs against the known-correct ones: a question whose one correct document was fully retrieved scores 100% recall; a question where only one of two correct documents was retrieved scores 50%, and so on. Averaging this across the entire dataset produces the overall Recall@k for the retriever — suppose that comes out to 67%.

No human judgment was needed anywhere in this process — a script compares two sets of document IDs and computes a ratio, which is exactly the kind of check a program can do reliably and cheaply. That's what makes this a **programmatic** eval: bringing in a human reviewer to do the same comparison would only add cost without adding accuracy.

## Acting on the result

A 67% recall score points toward specific fixes on the retriever side, rather than the generator or the application as a whole:

- **Improving the embedding model** — a weaker embedding model may not be capturing semantic meaning well enough to match queries to the right documents.
- **Query expansion** — passing the user's raw query through an LLM first to expand or rephrase it before sending it to the retriever, rather than sending it as-is.
- **Increasing k** — trying `k = 10` instead of `k = 5`, in case the correct documents are being retrieved but just outside the current cutoff.
- **Adding a reranker** — a document that shows up at rank 8 out of 10 candidates but never makes it into the top 5 can be pulled forward by reranking, directly improving recall without changing what gets retrieved in the first place.

## Recall is only one aspect of relevance

Recall@k captures one specific question: of the documents that should have been retrieved, how many actually were? It doesn't capture everything relevant to retrieval quality. Two related but distinct aspects worth knowing about:

- **Precision** — of the documents that *were* retrieved, how many were actually useful, rather than irrelevant noise mixed in alongside the correct ones?
- **Ranking quality** — even when all the correct documents are technically present somewhere in the retrieved set, whether they're properly ordered — surfaced near the top rather than buried near the bottom — matters separately, as seen earlier in the retriever-and-generator pipeline example.

## A distinction worth keeping separate: creating vs. executing an eval

Whether an eval pipeline is programmatic, human, or model-graded describes who or what *executes* the evaluation and produces the score — not who builds the golden dataset it runs against. Creating the golden dataset is a separate activity from running the eval, and it's carried out by a human regardless of which of the three methods ends up scoring the results. In the retriever example above, a human expert built the golden dataset by identifying the correct source documents for each question, even though the actual recall calculation that ran against that dataset was fully programmatic.

## A human example: rating chatbot helpfulness

Consider a general-purpose chatbot on top of a knowledge base — one that fields a wide range of questions rather than a single narrow task: when a course launches, what it costs, whether it comes with a certificate, how long it stays valid, and so on. The evaluation here is scoped to **application quality**, specifically **helpfulness** — not safety, not operations.

**Task and target.** The target is the entire application, not a single component or workflow. The task is to evaluate its helpfulness — whether the answer a user gets back is accurate, in the right tone, and complete in itself.

**Success criterion.** This is where things get harder than the retriever example. There's no universal, off-the-shelf metric for "helpfulness" the way there is for retrieval — it depends on the specific application. What gets defined instead is a **rubric**: a helpfulness scale from 1 to 5, where a 5 means the answer is accurate, complete, and delivered in the right tone; a 3 means it's partially helpful; and a 1 means it isn't helpful at all, or misses the point of the question entirely.

## Building the dataset: questions only

As before, the dataset is built by sampling 50 to 100 realistic questions — normal ones, difficult ones, edge cases, and anything else a user might plausibly ask. The key difference from the retriever example: this dataset has only **one column**, the question itself — there's no predefined "correct document ID" or "correct answer" to check against, because helpfulness isn't something that can be captured as a single expected string.

## Choosing the evaluation method: why human judgment is needed here

The next question is whether helpfulness can be measured programmatically — can a script look at a question and an answer and reliably output "70% helpful" or "90% helpful"? For a judgment this nuanced and subjective, the answer is no. A program can check whether an output exactly matches a known value; it can't judge tone, completeness, or whether an answer actually addressed what someone was asking. That makes the eval method here **human**.

## Running the human evaluation pipeline

The pipeline runs one question at a time: each question is sent to the chatbot, which generates an answer. A person then reads the question and the answer together and assigns a score based on the 1-to-5 rubric defined earlier. This repeats across the entire dataset, and the final helpfulness score for the system is the average of all the individual ratings.

## Why use more than one grader

A single grader's judgment is straightforward to trust, but using two or more graders on the same dataset serves a different purpose: it's a check on the rubric itself, not just on the chatbot. If two graders consistently agree on their scores, that's a sign the rubric is clear and unambiguous. If their scores diverge significantly on the same answers, that's a sign the rubric itself is ambiguous — different people are interpreting "helpful" differently — and it needs to be refined before the scores it produces can be trusted.

## Five types of human evaluation

Direct grading — reading an answer and scoring it against a rubric, as above — is only the simplest form human evaluation takes. In practice, humans contribute to LLM evaluation in five distinct ways:

1. **Direct grading and rating.** A person reads an output and scores it against a defined rubric. This is the most basic role — the human as grader — and it's what the example above walks through.
2. **Red teaming.** A group of people deliberately attacks a system — crafting jailbreaks, prompt injections, and adversarial or edge-case inputs — specifically trying to make it fail or produce harmful output. Before major LLM systems launch, dedicated red-teaming efforts try to break the system, and whatever failures they surface get fed back to the development team to fix. This kind of creative, adversarial probing isn't something automated evals can originate on their own.
3. **A/B testing.** Two versions of a system are put in front of real users in production, and their behavior — ratings, task completion, re-asking, escalation — becomes the verdict on which version performs better. Here, users are evaluating the application directly in production, rather than against a predefined dataset.
4. **Golden dataset and rubric creation.** Writing or verifying correct answers — gold document IDs, gold SQL, concept rubrics — and curating a test set to represent the real distribution of inputs is itself a form of evaluation, even though it happens before any scoring runs. Deciding that a specific document contains the correct answer to a specific question, as in the retriever example earlier, is already an evaluative judgment.
5. **Human-in-the-loop (HITL).** A human is placed inside the live production flow as a checkpoint — reviewing, approving, editing, or rejecting outputs before they reach the user. This gets triggered selectively, on the riskiest slice of traffic: low-confidence outputs, high-stakes situations, or anything flagged by an earlier automated or model-graded check. It's used when a case is too complex or ambiguous to trust to a programmatic check or an LLM judge alone.

## The trade-off with human evaluation

The clear advantage of using humans for evaluation is reliability — a human's judgment on nuanced, nonstandard, or high-stakes questions is generally more trustworthy than either a rigid program or a model, precisely because a human brain can weigh context and nuance in ways neither can yet.

The clear disadvantage is cost. Paying people to evaluate outputs doesn't scale to a system serving millions of queries, which makes pure human evaluation impractical for anything running at real production volume.

## Between programmatic and human: model-graded evaluation

This creates a real gap. Some things — like judging how helpful a chatbot's answer is — are too nuanced for a program to check mechanically, but evaluating them with humans at scale is too costly. The method that sits between the two, combining a program's speed and cost profile with something closer to a human's judgment, is the third category: **model-graded** (or **LLM-graded**) evaluation, where an LLM itself produces the verdict.

This is the most widely used of the three methods in practice — most LLM eval pipelines built today rely on it — and the specific technique behind it is commonly called **LLM-as-a-judge**: using an LLM to evaluate another system's output, the same way a human grader would, but without the cost or throughput limits that come with an actual person doing it.

## Summary: the three eval methods

1. **Programmatic / deterministic.** Code produces the verdict — an exact match, a regex, schema validation, or executing the output (running generated code or SQL) and comparing results. Fast, cheap, and fully reproducible, but only works where correctness is mechanically checkable.
2. **Human.** A person produces the verdict, rating, ranking, or comparing outputs directly. The slowest and most expensive method, but the gold standard for nuanced, subjective, or high-stakes judgments.
3. **Model-graded (LLM-as-judge).** A model produces the verdict, scoring or ranking output against a rubric. Used wherever there's no single correct answer but "good" can still be described — helpfulness, tone, groundedness — which is exactly the gap direct human grading is too costly to fill at scale.

## A model-graded example: automated grading for a competitive exam

Consider an exam-prep platform for a large, high-stakes competitive exam with three stages: a multiple-choice preliminary round, a subjective written round, and an interview. Auto-grading the multiple-choice round is trivial. The subjective written round is not — grading it properly requires subject-matter experts, and if tens of thousands of aspirants sit for a mock version of that exam, hiring enough experts to grade every paper becomes prohibitively expensive, cutting directly into how profitable running these mock tests can be.

Suppose a vendor offers a platform that claims to grade these subjective answers automatically, at any scale, against a defined rubric — for a fraction of the cost of human graders. Before adopting it, that platform itself needs to be evaluated: does it actually grade the way a human expert would?

## Task, target, and success criteria

**Task and target.** The target is the grading platform itself; the task is checking whether it evaluates subjective answers the way a human expert does.

**Success criterion.** The platform is considered successful if it grades answers the same way a human expert would. This is deliberately the same success criterion as the earlier chatbot example in spirit — matching human judgment — but here it plays out with a defined reference to compare against, rather than an open-ended rubric alone.

## Defining a rubric per question

Before any answers can be graded, each question needs its own rubric — the specific dimensions a strong answer should cover, decided by a subject-matter expert ahead of time. For a written exam with, say, three questions:

| Q ID | Question (abbreviated) | Max marks | Expected dimensions |
|---|---|---|---|
| q1 | "Ethical governance is impossible without administrative accountability." Discuss. | 15 | defines ethical governance & accountability · explains the link between them · gives mechanisms (e.g. RTI, social audit, oversight bodies) · cites examples/case studies · balanced conclusion |
| q2 | Examine the role of a state governor in centre–state relations. | 10 | constitutional role · discretionary powers · points of friction/misuse · relevant expert-committee recommendations · balanced view |
| q3 | "Federalism in this context is more cooperative than competitive." Critically analyze. | 15 | defines cooperative vs. competitive federalism · arguments for cooperative federalism · arguments for competitive federalism · recent tensions · reasoned stance |

This rubric is not itself the dataset — it's the scoring standard each answer to each question will be checked against.

## Building the golden dataset: human-graded answers

With the rubric in place, a small set of real aspirant answers — 50 to 100 is enough — gets collected and handed to a human subject-matter expert, who reads each one against the rubric for its question and assigns marks per dimension along with a total score. A few entries might look like this:

| Ans ID | Q ID | Answer summary | Dimensions covered | Human marks |
|---|---|---|---|---|
| a1 | q1 | Defines both terms, links them, discusses accountability mechanisms, cites an example, balanced conclusion. | 5 of 5 | 13/15 |
| a2 | q1 | Long, fluent essay on governance generally; never links to accountability, no mechanisms, no examples. | 1 of 5 | 4/15 |
| a3 | q1 | Covers the link and mechanisms well but no example and an abrupt ending. | 3 of 5 | 8/15 |
| a4 | q2 | Constitutional role, discretionary powers, and relevant recommendations covered; misses friction points and a conclusion. | 3 of 5 | 6/10 |
| a5 | q2 | Vague, mostly restates the question with heavy jargon, no constitutional grounding. | 0–1 of 5 | 1/10 |
| a6 | q3 | Defines both terms, strong two-sided argument with relevant examples, reasoned stance. | 4 of 5 | 12/15 |
| a7 | q3 | Only argues one side, ignores the competitive angle, no recent developments discussed. | 2–3 of 5 | 6/15 |

This is the golden dataset for this evaluation — a human evaluator's rubric-based marks for a modest, manageable set of real answers, not thousands of them.

## Choosing the evaluation method

Grading free-text subjective answers isn't something a program can do reliably — there's no exact string to match against, and the same idea can be expressed in countless different ways. That rules out a programmatic method. Using human graders at the scale this platform needs to operate at is exactly the cost problem the platform exists to solve, so that's ruled out too. That leaves **model-graded evaluation**: using an LLM to grade each answer against the same rubric a human would use, and checking how closely its marks track the human marks already collected.

## Running the evaluation: the judge prompt

Each stored answer is sent to an LLM configured to act as a grader, along with everything it needs to grade consistently: the question, its maximum marks, the exact rubric dimensions for that question, and the aspirant's answer. A prompt along these lines does the job:

```
You are grading an exam answer against an evaluation rubric.

QUESTION (max marks: {max_marks}): {question}
RUBRIC — expected dimensions and their marks:
{rubric_dimensions_for_this_question}
ASPIRANT ANSWER: {answer}

For each dimension, decide whether the answer genuinely addresses it
(and how well), then allocate marks. Do not reward verbosity,
keyword-stuffing, or confident assertions that lack substantiation.
Reward structure, relevant examples, and balanced argumentation.

Respond in JSON only:
{
  "dimensions": [
    {"dimension": "...", "addressed": true|false, "marks": <number>}
  ],
  "total_marks": <number>,
  "reasoning": "two-sentence justification"
}
```

Explicitly instructing the judge not to reward verbosity or keyword-stuffing matters here — a rubric-based judge can otherwise be fooled by an answer that mentions the right terms without actually engaging with them.

## Comparing human marks against judge marks

Running every stored answer through the judge produces a second column of marks, sitting directly alongside the human marks already collected for the same answers:

| Ans ID | Q ID | Human marks | Judge marks |
|---|---|---|---|
| a1 | q1 | 13 | 12 |
| a2 | q1 | 4 | 8 |
| a3 | q1 | 8 | 8 |
| a4 | q2 | 6 | 6 |
| a5 | q2 | 1 | 2 |
| a6 | q3 | 12 | 12 |
| a7 | q3 | 6 | 5 |

The success criterion — does the platform grade the way a human would — is hiding directly inside these two columns: the closer they track each other, the more the platform's grading matches human judgment.

## Turning that into a metric: Mean Absolute Error

The natural metric for how closely two sets of numeric scores track each other is **Mean Absolute Error (MAE)** — the average absolute difference between each pair of human and judge marks:

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |\,\text{human}_i - \text{judge}_i\,|$$

Applying this to the table above, with $n = 7$ answers:

$$\text{MAE} = \frac{|13-12| + |4-8| + |8-8| + |6-6| + |1-2| + |12-12| + |6-5|}{7} = \frac{1 + 4 + 0 + 0 + 1 + 0 + 1}{7} = \frac{7}{7} = 1.0$$

An MAE of 1.0 means the judge's marks deviate from the human's by an average of one mark — reasonably close, on a scale running up to 15. The goal going forward is to push this number toward zero: an MAE of 0 would mean the judge grades every answer exactly as the human did. Getting there typically involves the same levers as before — a stronger underlying model, a refined judge prompt, or a more precisely worded rubric — iterated on in the same loop as any other evaluation.

## Reference-based vs. reference-free evaluation

Two terms worth being able to tell apart, since they come up often when discussing eval design:

- **Reference-based evaluation.** A known-correct answer — or the specific things a correct answer must contain — is written down in advance for each test case, and the output is graded by comparing it against that reference.
- **Reference-free evaluation.** There's no predefined correct answer for each test case. The output's quality is judged directly, against a criterion or rubric that describes what a *good* answer looks like in general — a scale or standard, not a per-item correct answer.

The three worked examples covered so far split cleanly across this distinction:

- The **retriever recall** example is reference-based — the correct document IDs for each query were known in advance.
- The **chatbot helpfulness** example is reference-free — there was no predefined correct answer, only a 1-to-5 rubric describing what a helpful answer looks like in general, and a human judged each answer against that standard.
- The **exam-grading judge** example is reference-based — the human expert's marks serve as the known-correct reference the judge's marks are compared against.

A quick way to tell which is which: if the golden dataset includes a predefined correct answer for each item, it's reference-based; if it only includes a scale or standard to judge against, with no per-item answer to check, it's reference-free.